# Notebook 01 — Protein Structure Hierarchy

**Module:** 11 — Structural Biology  
**Tier:** 3 — Survey  
**Notebook:** 01 of 08  
**Time estimate:** 60 minutes

> A protein is a polypeptide chain that folds into a specific three-dimensional shape.
> That shape IS the function.
> Understanding structure levels is prerequisite for every other topic in this module.

---
## Step 1 — Motivation

Why does protein structure matter computationally? Because:
1. **Function is encoded in shape** — enzyme active sites, binding interfaces,
   allosteric sites are all 3D concepts
2. **AlphaFold2 is now central to biology** — you cannot participate in modern
   structural bioinformatics without understanding what a protein structure is
3. **Structure is the bridge between sequence (NB08–NB09) and function** —
   homology modelling, structure-based drug design, protein engineering all
   require this vocabulary

---
## Step 2 — Intuition

Think of a protein like a piece of origami:
- The **sequence** is the flat paper with a printed pattern (primary structure)
- Local creases form **secondary structures** (helices and sheets)
- The whole piece folds into a specific **3D shape** (tertiary structure)
- Multiple pieces assemble into a **complex** (quaternary structure)

Unlike origami, protein folding is driven by **thermodynamics**, not by hand —
the native fold is the conformation that minimizes free energy in aqueous solution.

---
## Step 3 — Biological Background

### The Four Levels of Protein Structure

**1. Primary structure — amino acid sequence**  
The linear sequence of 20 amino acids, connected by peptide bonds. The sequence
determines all higher levels (Anfinsen's dogma, 1973: the native fold is encoded
in the sequence alone).

**2. Secondary structure — local regular conformations**  
Repetitive patterns stabilized by backbone hydrogen bonds:
- **α-helix:** right-handed helix, 3.6 residues/turn, H-bond between C=O of
  residue $i$ and N-H of residue $i+4$
- **β-sheet:** parallel or antiparallel strands, H-bonds between backbone groups
  of distant residues
- **Loops/turns:** irregular regions connecting secondary structure elements

**3. Tertiary structure — full 3D fold**  
The spatial arrangement of all atoms. Stabilized by:
- Hydrophobic core (nonpolar residues buried away from water)
- Disulfide bonds (Cys–Cys)
- Hydrogen bonds, salt bridges, van der Waals interactions

**4. Quaternary structure — multi-subunit assembly**  
Two or more polypeptide chains forming a complex (e.g. haemoglobin = 4 chains,
ribosome = 80+ proteins + rRNA).

### Backbone Dihedral Angles

The protein backbone has **three bonds per residue**, each with rotational freedom:
- **φ (phi):** rotation around N–Cα bond (C′–N–Cα–C′)
- **ψ (psi):** rotation around Cα–C′ bond (N–Cα–C′–N)
- **ω (omega):** rotation around C′–N bond (Cα–C′–N–Cα); nearly always 180°
  because the peptide bond has partial double-bond character

φ and ψ define the backbone conformation. α-helices have φ ≈ −60°, ψ ≈ −40°;
β-sheets have φ ≈ −120°, ψ ≈ +130°. The Ramachandran plot (NB04) maps which
(φ, ψ) combinations are sterically allowed.

---
## Step 4 — Mathematical Explanation

**Dihedral angle definition:**  
Given four bonded atoms A–B–C–D, the dihedral angle θ is the angle between the
plane containing A–B–C and the plane containing B–C–D.

$$\theta = \text{atan2}\big((\mathbf{b}_1 \times \mathbf{b}_2) \cdot \hat{\mathbf{b}}_2 \cdot |\mathbf{b}_1|,\; \mathbf{b}_1 \cdot \mathbf{b}_2\big)$$

where $\mathbf{b}_1 = \mathbf{B} - \mathbf{A}$, $\mathbf{b}_2 = \mathbf{C} - \mathbf{B}$,
$\mathbf{b}_3 = \mathbf{D} - \mathbf{C}$.

A cleaner formulation via cross products:
$$\mathbf{n}_1 = \mathbf{b}_1 \times \mathbf{b}_2, \quad \mathbf{n}_2 = \mathbf{b}_2 \times \mathbf{b}_3$$
$$\theta = \text{atan2}\big((\mathbf{n}_1 \times \mathbf{n}_2) \cdot \hat{\mathbf{b}}_2,\; \mathbf{n}_1 \cdot \mathbf{n}_2\big)$$

**Protein folding thermodynamics:**  
$$\Delta G_{\text{fold}} = \Delta H - T \Delta S$$

Native proteins are only marginally stable: $\Delta G_{\text{fold}} \approx -10$ to $-60$ kJ/mol.
The hydrophobic effect drives burial of nonpolar residues ($\Delta H$ term);
backbone entropy opposes folding ($\Delta S$ term).

In [ ]:
# Step 6 — Python: Simulate an ideal alpha helix and compute backbone geometry
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ---- Amino acid properties (needed throughout the module) ----
AMINO_ACIDS = {
    'ALA': {'1letter': 'A', 'hydrophobic': True,  'mw': 89.09,  'charge': 0},
    'ARG': {'1letter': 'R', 'hydrophobic': False, 'mw': 174.20, 'charge': +1},
    'ASN': {'1letter': 'N', 'hydrophobic': False, 'mw': 132.12, 'charge': 0},
    'ASP': {'1letter': 'D', 'hydrophobic': False, 'mw': 133.10, 'charge': -1},
    'CYS': {'1letter': 'C', 'hydrophobic': True,  'mw': 121.16, 'charge': 0},
    'GLU': {'1letter': 'E', 'hydrophobic': False, 'mw': 147.13, 'charge': -1},
    'GLN': {'1letter': 'Q', 'hydrophobic': False, 'mw': 146.15, 'charge': 0},
    'GLY': {'1letter': 'G', 'hydrophobic': False, 'mw': 75.03,  'charge': 0},
    'HIS': {'1letter': 'H', 'hydrophobic': False, 'mw': 155.16, 'charge': 0},
    'ILE': {'1letter': 'I', 'hydrophobic': True,  'mw': 131.17, 'charge': 0},
    'LEU': {'1letter': 'L', 'hydrophobic': True,  'mw': 131.17, 'charge': 0},
    'LYS': {'1letter': 'K', 'hydrophobic': False, 'mw': 146.19, 'charge': +1},
    'MET': {'1letter': 'M', 'hydrophobic': True,  'mw': 149.21, 'charge': 0},
    'PHE': {'1letter': 'F', 'hydrophobic': True,  'mw': 165.19, 'charge': 0},
    'PRO': {'1letter': 'P', 'hydrophobic': False, 'mw': 115.13, 'charge': 0},
    'SER': {'1letter': 'S', 'hydrophobic': False, 'mw': 105.09, 'charge': 0},
    'THR': {'1letter': 'T', 'hydrophobic': False, 'mw': 119.12, 'charge': 0},
    'TRP': {'1letter': 'W', 'hydrophobic': True,  'mw': 204.23, 'charge': 0},
    'TYR': {'1letter': 'Y', 'hydrophobic': False, 'mw': 181.19, 'charge': 0},
    'VAL': {'1letter': 'V', 'hydrophobic': True,  'mw': 117.15, 'charge': 0},
}

def generate_ideal_helix(n_residues, rise_per_residue=1.5, radius=2.3, residues_per_turn=3.6):
    """Generate Cα coordinates for an ideal right-handed alpha helix."""
    coords = []
    for i in range(n_residues):
        angle = 2 * np.pi * i / residues_per_turn
        x = radius * np.cos(angle)
        y = radius * np.sin(angle)
        z = i * rise_per_residue
        coords.append([x, y, z])
    return np.array(coords)

def generate_ideal_sheet_strand(n_residues, rise_per_residue=3.3):
    """Generate Cα coordinates for an ideal extended beta strand."""
    coords = [[0, 0, i * rise_per_residue] for i in range(n_residues)]
    return np.array(coords)

helix_coords  = generate_ideal_helix(20)
strand_coords = generate_ideal_sheet_strand(10)

print('Ideal alpha-helix (20 residues):')
print(f'  Rise per turn: {1.5 * 3.6:.1f} Å')
print(f'  Length: {helix_coords[-1, 2]:.1f} Å')
print(f'  Mean Cα–Cα bond distance: ', end='')
ca_dists = np.linalg.norm(np.diff(helix_coords, axis=0), axis=1)
print(f'{ca_dists.mean():.2f} ± {ca_dists.std():.2f} Å')
print()
print('Ideal beta-strand (10 residues):')
print(f'  Length: {strand_coords[-1, 2]:.1f} Å')

In [ ]:
# Step 7 — Visualization: 3D helix, strand, and amino acid property wheel
fig = plt.figure(figsize=(16, 5))

# Panel A: 3D alpha helix
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot(helix_coords[:, 0], helix_coords[:, 1], helix_coords[:, 2],
          'o-', color='steelblue', ms=5, lw=1.5, alpha=0.9)
ax1.set_title('A. Ideal α-helix\n(Cα trace, 20 residues)')
ax1.set_xlabel('X (Å)'); ax1.set_ylabel('Y (Å)'); ax1.set_zlabel('Z (Å)')

# Panel B: Extended beta strand
ax2 = fig.add_subplot(132, projection='3d')
ax2.plot(strand_coords[:, 0], strand_coords[:, 1], strand_coords[:, 2],
          's-', color='tomato', ms=6, lw=1.5, alpha=0.9)
ax2.set_title('B. Ideal β-strand\n(Cα trace, 10 residues)')
ax2.set_xlabel('X (Å)'); ax2.set_ylabel('Y (Å)'); ax2.set_zlabel('Z (Å)')

# Panel C: Amino acid property wheel (hydrophobic vs. polar, charge)
ax3 = fig.add_subplot(133)
aas = list(AMINO_ACIDS.keys())
n_aa = len(aas)
angles = np.linspace(0, 2 * np.pi, n_aa, endpoint=False)
for i, aa in enumerate(aas):
    props = AMINO_ACIDS[aa]
    color = 'steelblue' if props['hydrophobic'] else ('tomato' if props['charge'] != 0 else 'lightgreen')
    x = 0.9 * np.cos(angles[i])
    y = 0.9 * np.sin(angles[i])
    ax3.scatter(x, y, c=color, s=100, zorder=3)
    ax3.text(1.1 * np.cos(angles[i]), 1.1 * np.sin(angles[i]),
              props['1letter'], ha='center', va='center', fontsize=8)
ax3.set_xlim(-1.5, 1.5); ax3.set_ylim(-1.5, 1.5)
ax3.axis('off')
ax3.set_title('C. Amino acid property wheel\nBlue=hydrophobic, Red=charged, Green=polar')
for label, color in [('Hydrophobic', 'steelblue'), ('Charged', 'tomato'), ('Polar', 'lightgreen')]:
    ax3.scatter([], [], c=color, s=60, label=label)
ax3.legend(fontsize=7, loc='lower right')

plt.suptitle('Module 11 NB01: Protein Structure Hierarchy', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('protein_structure_hierarchy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. What are the four levels of protein structure? Give one example of a stabilizing
   interaction at each level.
2. An alpha helix has 3.6 residues per turn and a rise of 1.5 Å per residue.
   How long (in Å) is a 50-residue helix?
3. Why does proline disrupt alpha helices? (*Hint: think about the φ angle and the
   cyclic side chain.*)
4. What is Anfinsen's dogma and what Nobel-Prize-winning experiment demonstrated it?

---
## Step 10 — Quiz

1. Name the three backbone dihedral angles and state the typical (φ, ψ) values for
   an α-helix and a β-sheet.
2. What stabilizes the hydrophobic core of a folded protein?
3. What is the typical ΔG of protein folding and what does this tell you about
   protein stability?
4. What is the difference between tertiary and quaternary structure?
5. Why is ω almost always 180°?

---
## Step 12 — Reflection

> *[In one paragraph: explain why knowing a protein's amino acid sequence is not
> sufficient to know its function, and why 3D structure is needed.]*

---
*Next: `02_experimental_structure_determination.ipynb`*